In [1]:
"""
checkpoint_manager.py
Simulates database checkpointing and checkpoint-based crash recovery.

Log record formats (database_journal.log), one per line:
    <T1 start>
    <T1, A, 100, 110>        <transaction, variable, old value, new value>
    <T1 commit>
    <T1 abort>
    <checkpoint T7, T8>      L = transactions active at the checkpoint
    <checkpoint>             L is empty

Simulated storage model
  * Volatile memory : buffer cache (cache, dirty pages) and the log buffer.
  * Stable storage  : the data disk (disk) and the log file database_journal.log.
  * Write-ahead logging: a log record reaches stable storage before the data page
    it describes, and a commit forces the log.
  * Checkpoint procedure:
        1. flush the log buffer to stable storage
        2. flush every dirty page to disk
        3. write <checkpoint L> (L = started, not yet committed/aborted) and flush it

Recovery (undo/redo, as in Task 1)
  * Backward scan from the end of the log: undo (restore old values) every update
    of a transaction that has no commit record (incomplete or aborted).
  * Forward scan: redo (write new values) every update of a committed transaction.
  * With a checkpoint, the scan stops once the checkpoint record has been reached
    AND the start record of every transaction in L has been seen. Without a
    checkpoint the scan runs back to the first record of the log.
"""

import json
import re
import sys

LOG_FILE = "database_journal.log"
INITIAL_DB = {"A": 100, "B": 200, "C": 300, "D": 400, "E": 500}

RE_STATUS = re.compile(r"^<\s*(\w+)\s+(start|commit|abort)\s*>$", re.IGNORECASE)
RE_UPDATE = re.compile(r"^<\s*(\w+)\s*,\s*(\w+)\s*,\s*(-?\d+)\s*,\s*(-?\d+)\s*>$")
RE_CKPT = re.compile(r"^<\s*checkpoint\s*([^>]*)>$", re.IGNORECASE)


# --------------------------------------------------------------------------
# Log parsing
# --------------------------------------------------------------------------
def parse_log(path):
    """Read a log file and return its records as a list of dicts."""
    records = []
    with open(path) as f:
        for lineno, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            m = RE_STATUS.match(line)
            if m:
                records.append({"type": m.group(2).lower(), "txn": m.group(1)})
                continue
            m = RE_UPDATE.match(line)
            if m:
                t, var, old, new = m.groups()
                records.append({"type": "update", "txn": t, "var": var,
                                "old": int(old), "new": int(new)})
                continue
            m = RE_CKPT.match(line)
            if m:
                active = [t.strip() for t in m.group(1).split(",") if t.strip()]
                records.append({"type": "checkpoint", "active": active})
                continue
            print(f"Warning: skipping malformed log line {lineno}: {line}", file=sys.stderr)
    return records


def get_active_transactions(log_path, upto=None):
    """
    Parse a log file and return the list L of active transactions, i.e. those
    that have a start record but no commit/abort record, in order of start.

    upto : number of log records to read (the "point" in the log). Records
           1..upto are considered. None means the whole log.
    """
    records = parse_log(log_path)
    if upto is not None:
        records = records[:upto]
    active = []
    for r in records:
        if r["type"] == "start":
            active.append(r["txn"])
        elif r["type"] in ("commit", "abort") and r["txn"] in active:
            active.remove(r["txn"])
    return active


# --------------------------------------------------------------------------
# Simulated database with volatile memory and stable storage
# --------------------------------------------------------------------------
class SimulatedDatabase:
    def __init__(self, log_path, initial):
        self.log_path = log_path
        open(log_path, "w").close()          # fresh stable log
        self.disk = dict(initial)            # stable storage: data pages
        self.cache = dict(initial)           # volatile: buffer cache
        self.dirty = set()                   # volatile: pages changed since last flush
        self.log_buffer = []                 # volatile: log records not yet on disk
        self.stable_log_count = 0            # records in the stable log
        self.logged_total = 0                # records ever generated
        self.active = []                     # active transactions, in start order
        self.writes = {}                     # txn -> [(var, old, new)]
        self.locks = {}                      # var -> txn (strict locking)
        self.expected = dict(initial)        # ground truth: committed work only
        self.checkpoints = []

    # -- log handling ------------------------------------------------------
    def _log(self, text):
        self.log_buffer.append(text)
        self.logged_total += 1

    def flush_log(self):
        """Move all buffered log records to the stable log file."""
        n = len(self.log_buffer)
        if n:
            with open(self.log_path, "a") as f:
                f.write("\n".join(self.log_buffer) + "\n")
            self.stable_log_count += n
            self.log_buffer = []
        return n

    # -- transaction operations -------------------------------------------
    def start(self, t):
        self.active.append(t)
        self.writes[t] = []
        self._log(f"<{t} start>")

    def write(self, t, var, new):
        holder = self.locks.get(var)
        if holder not in (None, t):
            raise RuntimeError(f"{t} cannot write {var}: locked by {holder}")
        old = self.cache[var]
        self.locks[var] = t
        self.cache[var] = new
        self.dirty.add(var)
        self.writes[t].append((var, old, new))
        self._log(f"<{t}, {var}, {old}, {new}>")

    def commit(self, t):
        self._log(f"<{t} commit>")
        self.flush_log()                     # commit forces the log
        for var, _old, new in self.writes[t]:
            self.expected[var] = new
        self._release(t)

    def abort(self, t):
        for var, old, _new in reversed(self.writes[t]):
            self.cache[var] = old
            self.dirty.add(var)
        self._log(f"<{t} abort>")
        self._release(t)

    def _release(self, t):
        for var in [v for v, h in self.locks.items() if h == t]:
            del self.locks[var]
        self.active.remove(t)

    # -- buffer manager ------------------------------------------------------
    def flush_page(self, var):
        """Write one page to disk early (a 'steal'); WAL flushes the log first."""
        self.flush_log()
        self.disk[var] = self.cache[var]
        self.dirty.discard(var)

    def checkpoint(self):
        flushed_log = self.flush_log()                 # 1. log -> stable storage
        pages = sorted(self.dirty)
        for var in pages:                              # 2. dirty pages -> disk
            self.disk[var] = self.cache[var]
        self.dirty.clear()
        active = list(self.active)                     # 3. checkpoint record
        self._log("<checkpoint " + ", ".join(active) + ">" if active else "<checkpoint>")
        flushed_log += self.flush_log()

        # Everything that lived in volatile memory must now be on stable storage.
        assert not self.log_buffer, "log buffer not empty after checkpoint"
        assert not self.dirty, "dirty pages remain after checkpoint"
        assert self.disk == self.cache, "disk and cache differ after checkpoint"
        assert self.stable_log_count == self.logged_total, "stable log incomplete"

        info = {"number": len(self.checkpoints) + 1, "record_no": self.logged_total,
                "active": active, "log_records_flushed": flushed_log,
                "pages_flushed": len(pages)}
        self.checkpoints.append(info)
        print(f"CHECKPOINT #{info['number']} at record {info['record_no']}: "
              f"L = {active}; flushed {flushed_log} log record(s) and "
              f"{len(pages)} dirty page(s) {pages}; volatile state is clean")

    # -- failure -----------------------------------------------------------------
    def crash(self):
        """System failure: everything in volatile memory is lost."""
        result = {"disk": dict(self.disk),
                  "lost_log_records": len(self.log_buffer),
                  "lost_dirty_pages": sorted(self.dirty)}
        self.log_buffer, self.cache, self.dirty = [], {}, set()
        return result


# Workload: (operation, arguments...). The checkpoints are taken while
# T7/T8 (first) and T11/T12 (second) are still active.
WORKLOAD = [
    ("start", "T1"), ("write", "T1", "A", 110), ("write", "T1", "B", 220), ("commit", "T1"),
    ("start", "T2"), ("write", "T2", "C", 330), ("commit", "T2"),
    ("start", "T3"), ("write", "T3", "D", 440), ("commit", "T3"),
    ("start", "T4"), ("write", "T4", "E", 550), ("commit", "T4"),
    ("start", "T5"), ("write", "T5", "A", 115), ("commit", "T5"),
    ("start", "T6"), ("write", "T6", "B", 225), ("write", "T6", "C", 335), ("commit", "T6"),
    ("start", "T9"), ("write", "T9", "A", 120), ("commit", "T9"),
    ("start", "T7"), ("write", "T7", "D", 450),
    ("start", "T8"), ("write", "T8", "E", 560),
    ("checkpoint",),                                   # L = [T7, T8]
    ("write", "T7", "B", 230), ("commit", "T7"),
    ("start", "T10"), ("write", "T10", "C", 345), ("commit", "T10"),
    ("abort", "T8"),
    ("start", "T13"), ("write", "T13", "B", 240), ("commit", "T13"),
    ("start", "T11"), ("write", "T11", "A", 130),
    ("start", "T12"), ("write", "T12", "E", 575),
    ("checkpoint",),                                   # L = [T11, T12]
    ("write", "T11", "D", 470), ("commit", "T11"),
    ("start", "T14"), ("write", "T14", "C", 999),
    ("flush_page", "C"),                               # uncommitted data reaches disk
    ("start", "T15"), ("write", "T15", "B", 250), ("commit", "T15"),
]


def run_workload(db, workload):
    for op in workload:
        getattr(db, op[0])(*op[1:])


# --------------------------------------------------------------------------
# Recovery
# --------------------------------------------------------------------------
def recover(log_path, disk, use_checkpoint=True):
    records = parse_log(log_path)
    n = len(records)
    db = dict(disk)

    ckpt_idx = None
    if use_checkpoint:
        for i in range(n - 1, -1, -1):             # most recent checkpoint
            if records[i]["type"] == "checkpoint":
                ckpt_idx = i
                break
    pending = set(records[ckpt_idx]["active"]) if ckpt_idx is not None else set()

    # Backward scan: undo updates of every transaction without a commit record.
    committed, undone = set(), []
    scanned_back, stop, i = 0, n - 1, n - 1
    while i >= 0:
        r = records[i]
        scanned_back += 1
        if r["type"] == "commit":
            committed.add(r["txn"])
        elif r["type"] == "update" and r["txn"] not in committed:
            db[r["var"]] = r["old"]
            undone.append((r["txn"], r["var"], r["old"]))
        elif r["type"] == "start":
            pending.discard(r["txn"])
        stop = i
        if ckpt_idx is not None and i <= ckpt_idx and not pending:
            break                                  # checkpoint reached, all L starts seen
        i -= 1

    # Forward scan from the stopping point: redo committed updates.
    redone, scanned_fwd = 0, 0
    for j in range(stop, n):
        scanned_fwd += 1
        r = records[j]
        if r["type"] == "update" and r["txn"] in committed:
            db[r["var"]] = r["new"]
            redone += 1

    return {"db": db, "backward": scanned_back, "forward": scanned_fwd,
            "stop_index": stop, "ckpt_index": ckpt_idx,
            "L": records[ckpt_idx]["active"] if ckpt_idx is not None else None,
            "undone": undone, "redone": redone}


def show_recovery(label, r, n):
    print(f"--- Recovery {label} ---")
    if r["ckpt_index"] is not None:
        print(f"Most recent checkpoint: record #{r['ckpt_index'] + 1}, L = {r['L']}")
    else:
        print("No checkpoint used: the whole log is scanned")
    print(f"Backward scan: records {n} -> {r['stop_index'] + 1} ({r['backward']} records read)")
    print("  UNDO :", ", ".join(f"{t}: {v}={old}" for t, v, old in r["undone"]) or "none")
    print(f"Forward scan : records {r['stop_index'] + 1} -> {n} ({r['forward']} records read)")
    print(f"  REDO : {r['redone']} update(s) re-applied")
    print("  Final state:", r["db"], "\n")


def main():
    stats_out = sys.argv[sys.argv.index("--stats-out") + 1] if "--stats-out" in sys.argv else None

    print("=== 1. Running workload ===")
    db = SimulatedDatabase(LOG_FILE, INITIAL_DB)
    run_workload(db, WORKLOAD)
    crash = db.crash()
    print(f"\nCRASH. Lost from volatile memory: {crash['lost_log_records']} log record(s), "
          f"unflushed pages {crash['lost_dirty_pages']}")
    print("Disk at crash :", crash["disk"])
    print("Expected state:", db.expected, "(committed work only)\n")

    records = parse_log(LOG_FILE)
    n = len(records)

    print("=== 2. get_active_transactions() ===")
    for upto in (5, 27, 41, None):
        label = f"first {upto} records" if upto else f"whole log ({n} records)"
        print(f"{label:>26}: L = {get_active_transactions(LOG_FILE, upto)}")
    ok = True
    for i, r in enumerate(records):
        if r["type"] == "checkpoint":
            computed = get_active_transactions(LOG_FILE, upto=i)
            good = computed == r["active"]
            ok &= good
            print(f"Checkpoint record #{i + 1}: logged L = {r['active']}, "
                  f"recomputed L = {computed} -> {'OK' if good else 'MISMATCH'}")
    assert ok, "checkpoint list does not match the transactions active at that point"
    print()

    print("=== 3. Crash recovery ===")
    without = recover(LOG_FILE, crash["disk"], use_checkpoint=False)
    with_cp = recover(LOG_FILE, crash["disk"], use_checkpoint=True)
    show_recovery("WITHOUT checkpoint", without, n)
    show_recovery("WITH checkpoint", with_cp, n)
    assert without["db"] == db.expected, "recovery without checkpoint is wrong"
    assert with_cp["db"] == db.expected, "recovery with checkpoint is wrong"
    print("Both recoveries produce the expected consistent state.\n")

    tw, to = with_cp["backward"] + with_cp["forward"], without["backward"] + without["forward"]
    print("=== 4. Log records scanned ===")
    print(f"{'':28}{'No checkpoint':>15}{'Checkpoint':>13}")
    print(f"{'Backward (undo) scan':28}{without['backward']:>15}{with_cp['backward']:>13}")
    print(f"{'Forward (redo) scan':28}{without['forward']:>15}{with_cp['forward']:>13}")
    print(f"{'Total records scanned':28}{to:>15}{tw:>13}")
    print(f"Reduction: {round(100 * (1 - tw / to), 1)}%")

    if stats_out:
        stats = {
            "total_records": n,
            "checkpoints": db.checkpoints,
            "without_checkpoint": {"start_record": without["stop_index"] + 1,
                                   "backward": without["backward"],
                                   "forward": without["forward"], "total": to},
            "with_checkpoint": {"start_record": with_cp["stop_index"] + 1,
                                "backward": with_cp["backward"],
                                "forward": with_cp["forward"], "total": tw,
                                "L": with_cp["L"],
                                "checkpoint_record": with_cp["ckpt_index"] + 1},
            "reduction_percent": round(100 * (1 - tw / to), 1),
            "disk_at_crash": crash["disk"],
            "lost_dirty_pages": crash["lost_dirty_pages"],
            "final_state": with_cp["db"],
        }
        with open(stats_out, "w") as f:
            json.dump(stats, f, indent=2)


if __name__ == "__main__":
    main()

=== 1. Running workload ===
CHECKPOINT #1 at record 28: L = ['T7', 'T8']; flushed 5 log record(s) and 5 dirty page(s) ['A', 'B', 'C', 'D', 'E']; volatile state is clean
CHECKPOINT #2 at record 42: L = ['T11', 'T12']; flushed 5 log record(s) and 4 dirty page(s) ['A', 'B', 'C', 'E']; volatile state is clean

CRASH. Lost from volatile memory: 0 log record(s), unflushed pages ['B', 'D']
Disk at crash : {'A': 130, 'B': 240, 'C': 999, 'D': 450, 'E': 575}
Expected state: {'A': 130, 'B': 250, 'C': 345, 'D': 470, 'E': 550} (committed work only)

=== 2. get_active_transactions() ===
           first 5 records: L = ['T2']
          first 27 records: L = ['T7', 'T8']
          first 41 records: L = ['T11', 'T12']
    whole log (49 records): L = ['T12', 'T14']
Checkpoint record #28: logged L = ['T7', 'T8'], recomputed L = ['T7', 'T8'] -> OK
Checkpoint record #42: logged L = ['T11', 'T12'], recomputed L = ['T11', 'T12'] -> OK

=== 3. Crash recovery ===
--- Recovery WITHOUT checkpoint ---
No checkpoi